# Travel Chat CLI

In [ ]:
%pip install -U transformers torch accelerate sentence-transformers rank-bm25 underthesea

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
%pwd

In [4]:
import os
import sys
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.load_document import load_documents
from core.retrieval.retriever import Retriever
from core.retrieval.context_builder import ContextBuilder
from core.retrieval.skip_retrieval import should_skip_retrieval
from chat.prompt import PromptBuilder

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.3
TOP_P = 0.9

print("Loading tokenizer/model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Loading retrieval resources...")
DOC_PATH = os.path.join(PROJECT_ROOT, "knowledge", "ho-chi-minh", "dinh-doc-lap")
documents = load_documents(DOC_PATH)
retriever = Retriever(strategy="hybrid")
context_builder = ContextBuilder(documents)
prompt_builder = PromptBuilder()

print("Ready.")

c:\Users\ADMIN\OneDrive\Desktop\latech\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer/model...


c:\Users\ADMIN\OneDrive\Desktop\latech\project\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 6703.68it/s]
Some parameters are 

Loading retrieval resources...


ValueError: Folder not exist: c:\Users\ADMIN\OneDrive\Desktop\latech\knowledge\ho-chi-minh\dinh-doc-lap

In [ ]:
def generate_answer(query: str, threshold: float = 0.8):
    query_clean = query.strip()
    if not query_clean:
        return "Query trong", [], ""

    skip = should_skip_retrieval(query_clean)

    results = []
    context = ""
    if not skip:
        results = retriever.search_threshold(query_clean, threshold=threshold)
        context = context_builder.build_text(results) if results else ""

    messages = prompt_builder.build_messages(query_clean, context)
    text_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True
        )

    new_tokens = generated[0][model_inputs.input_ids.shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return answer, results, context


def chat_cli(threshold: float = 0.8):
    print("Travel Chat CLI started. Type 'exit' to stop.")
    while True:
        q = input("\nUser: ")
        if q.strip().lower() in {"exit", "quit"}:
            print("Bye!")
            break

        answer, results, context = generate_answer(q, threshold=threshold)
        print("\nBot:", answer)
        print(f"[debug] docs={len(results)}, context_chars={len(context)}")


chat_cli(threshold=0.8)